# Six-band spectral mask: sweep every combination

Divide the original pulse’s **99.9%-energy bandwidth into six equal angular-frequency portions**. Each portion has field-amplitude options **0, 0.5 or 1**: **3⁶ = 729 complete masks**. Evaluate every combination, then rank feasible masks separately for **LP01 D_NL** and **LP01 D_IO**. Previous notebooks are preserved.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebook_helpers" / "run_library.py").is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook_helpers"))
from itertools import product

import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
from llm_bo.spectral_mask_six import run_experiment, load_run
from llm_bo.spectral_mask_six_report import build_report
from notebook_helpers.run_library import file_saved_run

SAVED_RUN = None  # Use an existing local campaign directory to reopen it.
if SAVED_RUN is None:
    run = run_experiment(max_minutes=120)
else:
    run = load_run(SAVED_RUN)
SAVED_RUN = run.directory
file_saved_run(SAVED_RUN, "Spectral amplitude mask", "Exhaustive grid sweep")
print("Run directory:", SAVED_RUN)
print("Stop reason:", run.result["stop_reason"])
report_ready = all(run.result.get("validation", {}).get(metric, {}).get("selected_refined")
                   for metric in ("D_NL", "D_IO"))
report_ready = report_ready and len(run.result["sweep"]) == 729
if report_ready:
    build_report(SAVED_RUN)
else:
    print("The time budget ended before the full report was available. Measurements are in state.json.")

## 1. Keep the physics; use six controls

**100 kW before masking, 200 fs Gaussian; three equal-amplitude modes with zero relative phase.** Keep the 2.86 m fiber, 52.5 µm core radius, NA 0.1, 1379.2 nm carrier and the original material/dispersion parameters. Apply one shared amplitude mask to every mode, with **no renormalization**. Frequencies outside the controlled bandwidth remain unchanged.

Band 1 has the lowest angular-frequency detuning; band 6 has the highest. Equal angular-frequency widths are not equal wavelength widths or equal-energy portions. **0.5 amplitude gives 0.25 power** within that portion. Even the all-zero six-factor mask retains the small outside-band tails.

In [ ]:
if report_ready:
    spec = json.loads((SAVED_RUN / "run.json").read_text())
    display(pd.read_csv(SAVED_RUN / "band_definitions.csv"))
    problem_card = {
        "objectives": "minimize LP01 D_NL and D_IO separately",
        "brightness": "all-mode output photons / unmasked baseline >= 0.50",
        "search grid": "Nt = 16384; dz = 5 mm; original time window",
        "budget": "729 combinations; at most 735 new pairs including checks; 120 minutes",
    }
    display(pd.Series(problem_card, name="Setup").to_frame())

## 2. Enumerate all 729 masks

Each row is one complete six-factor mask. The sixth factor changes fastest. Every factor takes each value exactly **243 times**, and every possible joint combination appears once. This is a complete finite sweep; no random sampling, local search or feasibility restoration is added.

In [ ]:
if report_ready:
    levels = (0.0, 0.5, 1.0)
    masks = np.array(list(product(levels, repeat=6)))

    assert masks.shape == (729, 6)
    assert len(np.unique(masks, axis=0)) == 729
    assert np.array_equal(masks, np.load(SAVED_RUN / "all_masks.npy"))
    print("729 masks × 6 factors; first and last masks:")
    display(pd.DataFrame(masks[[0, -1]], columns=[f"factor_{i+1}" for i in range(6)]))

## 3. Evaluate every combination and keep feasible minima

One new evaluation runs the same input through a **linear** fiber and a **nonlinear GNLSE**, then computes both LP01 scores. `D_NL` compares the two outputs; `D_IO` compares input with nonlinear output. Both use normalized complex overlap, with no fitted time shift. The 50% brightness floor uses **all-mode output photons**, relative to the unmasked nonlinear baseline at the matching finer grid.

Every mask is simulated, including masks below the floor. Cache hits add no simulations; the all-ones sweep point reuses the baseline. All inputs, fields, scores, rejected-by-brightness results and elapsed times are saved. The execution logic is:

```python
# baseline measures the unmasked reference on both propagation-step grids.
campaign.baseline()
for index, mask in enumerate(masks):
    # sample records one mask and updates both feasible minima.
    campaign.sample(index, mask)
# validate rechecks the selected masks with finer propagation and time grids.
campaign.validate()
```

The following cell replays the selection from saved measurements:

In [ ]:
if report_ready:
    best = {objective: run.result["baseline"] for objective in ("D_NL", "D_IO")}
    for trial in run.result["sweep"]:
        for objective in best:
            if trial["feasible"] and trial[objective] < best[objective][objective]:
                best[objective] = trial

    summary = json.loads((SAVED_RUN / "sweep_summary.json").read_text())
    print(summary["feasible"], "feasible masks out of", summary["total_combinations"])
    print("Stop:", run.result["stop_reason"])
    display(pd.read_csv(SAVED_RUN / "selected_masks.csv"))

### Sweep path and elapsed time

The path figure shows both objectives, measured brightness and every mask in enumeration order. Grey crosses fail brightness; the line tracks the best feasible score. Diamonds mark the initial baseline and final incumbent. Abrupt changes follow the enumeration order, not gradient-descent steps. Finer-check points are separate from the search-grid path.

In [ ]:
if report_ready:
    for name in ("sweep_path", "convergence_elapsed", "selected_masks"):
        display(Image(filename=str(SAVED_RUN / f"{name}.png")))

## 4. Check the selected masks

First reduce the propagation step from **5 mm to 1 mm** at Nt = 16384. Then use **Nt = 65536 and twice the time window**, still at 1 mm. The absolute target-score tolerance is **2×10⁻⁴**. Brightness must remain at least 50% on both finer checks. A failed or missing check keeps that branch **preliminary**.

Each branch’s acceptance applies to its optimized metric; the other reported metric is a diagnostic. All 729 combinations are covered at the search resolution. Checking only the finalists does not establish the full ranking at finer resolution or an optimum over continuous factors.

**D_NL: preliminary.** Selected time-grid change: 0.001330; refined brightness: 50.194%.

**D_IO: preliminary.** Selected time-grid change: 0.001056; refined brightness: 50.038%.

In [ ]:
if report_ready:
    display(pd.read_csv(SAVED_RUN / "selected_results.csv"))
    display(pd.read_csv(SAVED_RUN / "validation_checks.csv"))
    display(pd.read_csv(SAVED_RUN / "reported_metric_grid_changes.csv"))

## 5. Compare the input, linear output and nonlinear output

Columns show the unmasked baseline, D_NL-selected mask and D_IO-selected mask, all on the same finer time grid. Time-domain power appears above wavelength spectral energy density. Each case has its own matched input/linear/nonlinear fields. Units are **kW** and **nJ/nm**, including the frequency-to-wavelength Jacobian; there is no per-curve normalization or time alignment. Input and linear spectra can overlap.

**LP01**, the optimized mode:

In [ ]:
if report_ready:
    display(Image(filename=str(SAVED_RUN / "pulse_comparison_lp01.png")))

The other two modes and the spatially integrated all-mode power:

In [ ]:
if report_ready:
    for mode in ("lp11_cos", "lp11_sin", "all_modes"):
        display(Image(filename=str(SAVED_RUN / f"pulse_comparison_{mode}.png")))

Full-window LP01 traces show low-power tails beyond the central ±2 ps view:

In [ ]:
if report_ready:
    display(Image(filename=str(SAVED_RUN / "pulse_comparison_lp01_full_window.png")))

## 6. Measured runtime and compute cost

One objective pair is **one nonlinear GNLSE plus one matched linear solve**. The active run timer covers setup, baselines, the sweep, validation and worker shutdown. Calendar duration is shown separately: the two clocks differed during this run, and the cause of that gap was not measured. All resource plots use calendar time and shade sampling gaps; convergence uses active run time. Solver and controller timers disagreed across the gaps, so setup overhead is marked unmeasured. Report generation and notebook export are timed separately.

CPU/memory sampling starts before the first solve. Memory is a sampled process-tree peak. **The solver uses the CPU**; the recorded device-wide GPU activity includes other applications. Storage is a dated inventory, with GB = 10⁹ bytes. Electricity and monetary cost were not measured. Implementation before campaign creation was not timed. The pre-run runtime estimate is kept separate from measured costs.

In [ ]:
if report_ready:
    cost = json.loads((SAVED_RUN / "run_costs.json").read_text())
    display(pd.Series(cost, name="Measured cost").to_frame())
    display(Image(filename=str(SAVED_RUN / "measured_resources.png")))